# HealthPredict · Henry M4

Versión reproducible del ejercicio. Requiere los CSV originales; no sustituye resultados por datos demo.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train.py").is_file())
sys.path.insert(0, str(ROOT))
from train import prepare_data, fit_evaluate
DATA = ROOT / "data"

## 1. Datos y unión validada
Se excluyen nombres e identificadores del modelo. Se rechazan claves ambiguas.

In [ ]:
visits = pd.read_csv(DATA / "healthpredict_visits.csv")
patients = pd.read_csv(DATA / "healthpredict_patients.csv")
extra_path = DATA / "pacientes2.csv"
extra = pd.read_csv(extra_path) if extra_path.exists() else None
data = prepare_data(visits, patients, extra)
data.drop(columns=["patient_id"]).describe(include="all")

## 2. Entrenamiento y evaluación
Las visitas de un paciente pertenecen a una sola partición. Imputación y escalado se ajustan dentro del pipeline.

In [ ]:
model, report, train_idx, test_idx = fit_evaluate(data)
assert set(data.iloc[train_idx].patient_id).isdisjoint(data.iloc[test_idx].patient_id)
report

## 3. Interpretación
Las métricas corresponden al holdout por paciente de este dataset académico. No representan validación clínica ni una evaluación en producción.